# Start Here: Fine-Tuning LLMs with Unsloth Studio on AMD Instinct MI300X

Welcome. This is the landing notebook for the workshop. If you have never
fine-tuned a language model before, you are in the right place. By the end
you will understand what fine-tuning is, the main techniques (SFT, LoRA,
QLoRA, GRPO), what training data looks like, and you will have launched
**Unsloth Studio**, a friendly point-and-click app, on a real AMD GPU in
the cloud.

No prior deep-learning experience is required. Read top to bottom, run the
occasional cell, and follow the boxed steps.

---

## What you will do

1. Learn the core ideas of fine-tuning (a short, plain-English tour).
2. Launch Unsloth Studio on your AMD Dev Cloud GPU session.
3. Open the Studio in your browser through your secure Dev Cloud link.
4. Run your first fine-tune, all from a visual interface.

The step-by-step launch instructions are in the section
**"Launch Unsloth Studio"** further down. If you just want to get going,
jump there. If you want to understand *what* you are doing first, keep reading.


---

# Part 1: The Concepts (plain English)

## What is a large language model (LLM)?

An LLM is a neural network trained on a huge amount of text so that it can
predict the next word (technically, the next *token*) given what came before.
That simple skill, done at scale, is enough to produce models that can chat,
summarize, write code, and reason.

Models you download from the Hugging Face Hub (Llama, Qwen, Mistral, and
friends) are called **base** or **instruct** models. They are generalists.

## What is fine-tuning, and why do it?

Fine-tuning means taking one of those pre-trained models and continuing to
train it a little more on **your** data, so it adapts to **your** task,
domain, tone, or format.

You fine-tune when you want the model to:

- Speak in a specific style or persona (your brand voice, a support agent).
- Know a specific domain (legal, medical, your product docs).
- Follow a strict output format (always return JSON, always cite sources).
- Do a narrow task extremely well and cheaply on a smaller model.

Fine-tuning is often cheaper and more reliable than stuffing everything into
a giant prompt, and it lets a small model punch above its weight.

![Fine-tuning takes a pre-trained base model, adds your data, and produces a model specialized to your task](assets/01-finetuning-overview.png)

*Fine-tuning: a general base model plus your data becomes a specialist.*

## The two big families of fine-tuning

### 1. Supervised Fine-Tuning (SFT)

You show the model **examples of the right answer**. Each training example is
an input (a question or instruction) paired with the ideal output. The model
learns to imitate those outputs. This is the workhorse of fine-tuning and
where almost everyone starts.

Think of it as: *"Here are 1,000 questions and the exact answers I want. Learn
to answer like this."*

### 2. Reinforcement / preference methods (GRPO, DPO, PPO)

Instead of one gold answer, you steer the model with a **reward signal** or
**preferences** (this answer is better than that one). The model explores and
is nudged toward higher-reward behavior.

**GRPO** (Group Relative Policy Optimization) is a modern, efficient RL method
popularized by the DeepSeek work. It is great for teaching **reasoning**: the
model generates several candidate answers, they are scored by a reward
function (for example, "did it get the math right?"), and the model is pushed
toward the better ones. No human-written gold answer needed for every case,
just a way to score outputs.

Rule of thumb: **start with SFT.** Reach for GRPO when you want to improve
reasoning or optimize for a reward you can measure, after you have a decent
SFT baseline.

![SFT learns from gold input-output examples; GRPO scores several candidate answers and rewards the best](assets/02-sft-vs-grpo.png)

*The two families: SFT imitates gold answers; GRPO rewards the best of several tries.*

## Full fine-tuning vs LoRA vs QLoRA (the memory trick)

Modern models have billions of parameters. Updating **all** of them ("full
fine-tuning") needs a lot of GPU memory and time. Two ideas make this
practical on a single GPU:

### LoRA (Low-Rank Adaptation)

Instead of updating the whole model, LoRA **freezes** the original weights and
injects tiny trainable "adapter" matrices into a few layers. You train only
those small adapters, often less than 2 percent of the parameters. You get
most of the quality of full fine-tuning for a fraction of the memory, and the
adapter file is small (a few tens of MB) and easy to share.

### QLoRA (Quantized LoRA)

QLoRA goes one step further: it **quantizes** the frozen base model to 4-bit
(so it takes roughly a quarter of the memory) and then trains LoRA adapters on
top. This is what lets you fine-tune surprisingly large models on a single
consumer or workstation GPU.

The AMD Instinct MI300X gives you a massive **192 GB** of HBM3 memory per GPU,
so memory is rarely the constraint here. A 4-bit Qwen3-4B QLoRA run needs only
about **5.6 GB**, which barely registers. On the MI300X you can comfortably go
much bigger: fine-tune larger models (for example a Llama 3.x 70B in 4-bit),
use longer sequence lengths, or bump up the batch size for faster training.
The same QLoRA trick still applies, you just have far more headroom.

| Method | Trains | Memory | When to use |
| --- | --- | --- | --- |
| Full fine-tune | 100 % of weights | Very high | You have big GPUs and lots of data |
| LoRA | Small adapters (~1-2 %) | Medium | Great default, adapters are portable |
| QLoRA | Adapters on a 4-bit base | Low | Best for single-GPU / limited VRAM |

![Full fine-tuning trains all weights (high memory); LoRA freezes the base and trains small adapters; QLoRA quantizes the base to 4-bit then trains adapters (low memory)](assets/03-full-lora-qlora.png)

*The memory trick: freeze the base (LoRA) and quantize it to 4-bit (QLoRA) to fit on one GPU.*

## What does training data look like?

For **SFT**, the most common shape is a list of examples, each with an
instruction, an optional input, and the desired output. This is the classic
"Alpaca" format:

```json
{"instruction": "Translate to French.", "input": "Good morning", "output": "Bonjour"}
{"instruction": "Summarize in one line.", "input": "<article text>", "output": "<summary>"}
```

Chat-style datasets instead use a list of messages with roles:

```json
{"messages": [
  {"role": "user", "content": "What is QLoRA?"},
  {"role": "assistant", "content": "QLoRA is 4-bit quantization plus LoRA adapters..."}
]}
```

For **GRPO**, you mainly need prompts plus a **reward function** that scores
generated answers (for example, +1 if the final number matches the known
answer). The gold text is optional.

Quality beats quantity. A few hundred to a few thousand clean, consistent
examples usually beats a huge, noisy pile. In Unsloth Studio you can point at
a Hugging Face dataset or upload your own JSON/JSONL in these shapes.

## Where Unsloth fits

[Unsloth](https://github.com/unslothai/unsloth) is a library that makes
fine-tuning **much faster and much lighter on memory** through hand-optimized
kernels, with no loss in accuracy. It supports LoRA and QLoRA out of the box
for popular model families.

**Unsloth Studio** is the graphical app on top of it. Instead of writing
training code, you:

1. Pick a model.
2. Pick or upload a dataset.
3. Choose SFT or GRPO and set a few sliders (LoRA rank, steps, learning rate).
4. Click **Train** and watch the loss curve live.
5. Chat with your fine-tuned model to compare it against the base.

That is the whole loop, and you will do it yourself in a few minutes. Under
the hood it is the exact same SFT / LoRA / QLoRA you just read about.

---

# Part 2: Launch Unsloth Studio on AMD Instinct MI300X

You should already be inside a running **AMD Instinct MI300X** GPU session with
this notebook open (that is how you are reading this). On the AMD Dev Cloud you
do not provision anything by hand: opening this notebook from
`notebooks.amd.com` automatically spins up an MI300X GPU instance for you in
the background. If you are reading this, your GPU is already allocated and
ready.

The launch is just two moves: **run one cell**, then **open the secure
link it builds**.


![Run one cell to build your secure Studio link, then open it in your browser](assets/05-launch-flow.png)

*Two moves: run the cell, then open the secure link it builds.*


## How to use this notebook

Unsloth Studio is a no-code web UI for fine-tuning. On this AMD Dev Cloud
MI300X session it is **already running**, it starts automatically when
your session boots. You don't install or launch anything.

The next two steps get you into the Studio UI:

1. Run one cell to build your secure browser link.
2. Open that link in your browser.

That's it. (A terminal is only needed for the optional "restart Studio
yourself" path at the very end.)


## Step 1: Get your Studio link (Studio is already running)

You do **not** need to start anything. On this AMD Dev Cloud MI300X
session, Unsloth Studio launches automatically when your GPU session
boots.

Just run the cell below. It builds your **secure browser link** to the
Studio (served straight through the AMD Dev Cloud portal you are already
signed in to) and checks that the Studio backend is up.

> If it says "not ready yet", give it ~15 seconds after your session
> starts and run the cell again.


In [ ]:
# ============================================================================
#  Get your Unsloth Studio link  --  just run this cell (Shift+Enter)
# ----------------------------------------------------------------------------
#  Unsloth Studio is ALREADY RUNNING: the MI300X image starts it automatically
#  when your GPU session boots. You do NOT launch anything yourself.
#
#  On AMD Dev Cloud you reach Studio through your notebook's own secure address
#  (the Jupyter "server-proxy"), so it opens in your browser with no extra
#  login, no tunnel, and no copy-pasting of IPs. This cell builds that link.
# ============================================================================
import os, re, json, glob, urllib.request

STUDIO_PORT = 8890  # the port Studio listens on inside your session

def _studio_healthy(base):
    """Return True if the Studio backend answers on a given local base URL."""
    try:
        with urllib.request.urlopen(base + "/api/health", timeout=5) as r:
            return json.loads(r.read().decode()).get("status") == "healthy"
    except Exception:
        return False

def _find_studio_port():
    """Studio normally runs on 8890; fall back to scanning if that moved."""
    if _studio_healthy(f"http://127.0.0.1:{STUDIO_PORT}"):
        return STUDIO_PORT
    for p in (8888, 8889, 8891, 7860):
        if _studio_healthy(f"http://127.0.0.1:{p}"):
            return p
    return STUDIO_PORT  # best guess; the link may need a few seconds after boot

def _proxy_url(port):
    """Build the AMD Dev Cloud portal URL that proxies to Studio.

    JUPYTER_SERVER_URL looks like:
        http://0.0.0.0:8888/jupyter-user-<id>/
    and the public host is in BASE_URL (e.g. notebooks.amd.com). We turn that
    into:  https://<host>/jupyter-user-<id>/proxy/<port>/
    """
    server_url = os.environ.get("JUPYTER_SERVER_URL", "")
    m = re.search(r"(/jupyter-user-[^/]+)/?", server_url)
    prefix = m.group(1) if m else ""
    host = os.environ.get("BASE_URL", "").strip().rstrip("/")
    if host and not host.startswith("http"):
        host = "https://" + host
    if prefix and host:
        return f"{host}{prefix}/proxy/{port}/"
    if prefix:  # relative link still works from inside the portal
        return f"{prefix}/proxy/{port}/"
    return None

# --- Fallbacks for the standalone Docker image (not AMD Dev Cloud) ----------
def _read(path):
    try:
        with open(path) as fh:
            return fh.read()
    except OSError:
        return ""

def _scan_for_cloudflare():
    blob = _read("/workspace/CONNECTION_INFO.txt")
    for pat in ("/workspace/studio_launch.log",
                "/root/.unsloth/studio/logs/server/server-*.log"):
        for path in sorted(glob.glob(pat)):
            blob += _read(path)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", blob)
    return m.group(0) if m else None

# --- Build the best available link ------------------------------------------
port = _find_studio_port()
proxy = _proxy_url(port)
cloudflare = _scan_for_cloudflare()

print("=" * 68)
print("  Unsloth Studio  --  how to open it")
print("=" * 68)
if proxy:
    print("  Open Studio in your browser (AMD Dev Cloud secure link):")
    print(f"      {proxy}")
    print()
    print("  Tip: click the link above, or copy it into a new browser tab.")
    print("  You are already signed in through the AMD Dev Cloud portal, so")
    print("  Studio opens directly. No password or tunnel needed.")
elif cloudflare:
    print("  Open Studio in your browser (Cloudflare secure link):")
    print(f"      {cloudflare}")
else:
    print("  Studio link is not ready yet.")
    print("  Wait ~15 seconds after your session starts, then re-run this cell.")
    print(f"  (Looking for the Studio backend on port {port}.)")

print()
print(f"  Studio backend health on this session: "
      f"{'healthy' if _studio_healthy(f'http://127.0.0.1:{port}') else 'starting up...'}")
print("=" * 68)


## Step 2: Open the Studio in your browser

From the output above, **click the secure link** (or copy it into a new
browser tab). Because it is served through the AMD Dev Cloud portal you
are already signed in to, the Studio UI opens directly, no separate
password or login screen.

> Tip: the first time you open it, the UI can take a few seconds to load
> while the frontend initializes. If it looks blank, wait a moment and
> refresh once.


## Advanced: restart Studio yourself (optional)

You normally never need this, Studio is already running. But if you ever stop
it or want a fresh instance, open a terminal
(**File -> New -> Terminal**) and run:

```bash
unsloth studio -H 0.0.0.0 -p 8888 --cloudflare
```

Note: if Studio is *still* running, this command detects the busy port and
starts a **second** instance on 8889 instead. That second port is not exposed
outside the container, so stick to one instance and just re-run the cell above
to fetch its link.


---

# Part 3: Your first fine-tune inside the Studio

Once the Studio UI is open, here is the flow to follow. Everything maps back
to the concepts in Part 1.

![The Studio loop: pick a model, pick data, choose SFT or QLoRA, train and watch the loss drop, then compare against the base](assets/04-studio-loop.png)

*The whole loop in the Studio UI, the same SFT / LoRA / QLoRA under the hood.*

1. **Choose a model.** Start small, for example a 4-bit Qwen3-4B or a Llama
   3.x 8B instruct. Small models train fast and are perfect for learning.

2. **Choose a dataset.** Either search the Hugging Face Hub from the UI or
   upload your own JSON/JSONL in the Alpaca or chat format from Part 1. A
   small slice (500 to 1,000 rows) is plenty for a first run.

3. **Pick the method.** For your first run choose **SFT** with **LoRA/QLoRA**
   (QLoRA is the memory-friendly default). Try **GRPO** later once SFT feels
   familiar.

4. **Set a few knobs.** Sensible starting points: LoRA rank `r=16`, a short
   run of 30 to 60 steps for a demo, learning rate around `2e-4`. Defaults in
   the Studio are usually fine.

5. **Click Train.** Watch the **loss curve** go down. A falling loss means the
   model is actually learning from your data (not a no-op).

6. **Test it.** Use the chat / compare panel to talk to your fine-tuned model
   and compare it against the base model on the same prompt. This is the
   payoff: you can *see* the behavior change.

7. **Save / export.** Save the LoRA adapter (small, portable) or export a
   merged model / GGUF if you want to run it elsewhere.

That is a complete fine-tuning cycle. Congratulations, you just trained a
model on an AMD GPU.

---

## How to choose your dataset (Train > Datasets)

The Datasets picker bundles **two separate decisions**: *where* the data
comes from (the source) and *what task and format* it is for (which decides
what your fine-tune actually learns). Get both right and training just works.

### Decision 1: the source (where it loads from)

- **Hugging Face Hub** : search and pull a public dataset by name (for
  example `mlabonne/FineTome-100k`). Best when a well-known dataset already
  fits your task. Two gotchas on AMD Dev Cloud: the search box is *debounced*,
  so type a real keystroke and wait a moment for results; and the instance
  may be in Hugging Face *offline* mode, so a non-cached dataset can fail with
  `OfflineModeIsEnabled`. If that happens, switch to Local.
- **Local / Upload** : upload your own `JSON` / `JSONL` file. This is the
  reliable path on AMD Dev Cloud and the one to use for your own data.

### Decision 2: the task and format (what you are teaching)

This is the part that matters most. Match the dataset shape to the method:

| Goal | Method | Dataset type | Row format |
| --- | --- | --- | --- |
| Answer in your style / format, or teach domain Q and A | SFT | Instruction (Alpaca) | `{instruction, input, output}` |
| Better general multi-turn chat assistant | SFT | Chat / conversation (ShareGPT) | `{messages: [{role, content}, ...]}` |
| Improve step-by-step reasoning / math | GRPO | Prompts + reward function | prompts (gold answer optional) |

- **Instruction (Alpaca) format** is the simplest to build yourself. `input`
  can be empty. Great default for a first custom fine-tune.
- **Chat (ShareGPT) format** preserves system / user / assistant roles; use it
  for multi-turn assistants. `FineTome-100k` is a popular example.
- **GRPO** needs mainly prompts plus a way to *score* answers (for example, +1
  if the final number is correct, as with GSM8K). Reach for it after SFT.

### Quick decision shortcut

- Want the model to answer in your style / format, or learn your domain Q and
  A -> **SFT + Alpaca** (upload your own JSONL). Start here.
- Want a better general chat assistant -> **SFT + a chat dataset** like
  `FineTome-100k`.
- Want stronger reasoning / math -> **GRPO + a scorable dataset**.
- Just want to see it work end to end -> pick the small preset / example
  dataset (or a 500 to 1,000 row slice). You only need enough to watch the
  loss drop.

### The one critical rule

**The format must match the method.** An SFT run expects instruction/output or
messages rows; feed it the wrong shape and training either errors out or
learns nothing (a flat loss curve). If you upload your own file for SFT, keep
it to the exact Alpaca fields:

```json
{"instruction": "Summarize in one line.", "input": "<article text>", "output": "<summary>"}
{"instruction": "Translate to French.", "input": "Good morning", "output": "Bonjour"}
```

One JSON object per line for JSONL. `input` may be an empty string when the
instruction is self-contained.

## Good first experiments

- **Style transfer:** fine-tune on 300 examples written in a specific tone
  and watch the model adopt it.
- **Format enforcement:** train it to always answer in strict JSON.
- **Tiny domain expert:** feed it Q and A pairs from your product docs.
- **Reasoning with GRPO:** once comfortable, try a math dataset with a reward
  that checks the final answer.

## Troubleshooting

| Symptom | Fix |
| --- | --- |
| Studio link does not open | Wait 10-20 s and refresh once; the backend needs a moment after boot. |
| First chat response times out | Cold model load. Retry; the warm model responds fast. |
| Out-of-memory during training | Use a smaller model, enable QLoRA (4-bit), lower batch size or sequence length. |
| Loss not decreasing | Raise learning rate a little, train more steps, or check that your dataset format is correct. |
| `unsloth: command not found` | You are in the wrong environment; confirm you are in the AMD Dev Cloud MI300X session shell. |


---

# Next steps and deeper dives

When you want to go beyond the point-and-click Studio and see the actual
training code, explore the other notebooks in this repo:

- `qwen3_qlora_smoketest.ipynb` : a fully verified, end-to-end **QLoRA** run
  on AMD RDNA3 silicon. Every cell proves a real capability (GPU matmul, a
  real loss drop, post-training inference). Great for seeing the code behind
  the Studio.
- `llama-3.1-8b-grpo-unsloth.ipynb` : a **GRPO** reasoning example.
- `llama-3.2-vision-finetune-unsloth.ipynb` : fine-tuning a **vision** model.
- `model-finetuning-on-radeon.ipynb` : the Radeon working notebook (the code
  is portable; it runs on MI300X too).
- `qwen3-qlora-radeon-w7900-notes.md` : setup notes, gotchas, and measured
  numbers you can reuse as slide material.

## Learn more

- Unsloth: https://github.com/unslothai/unsloth
- Unsloth docs: https://docs.unsloth.ai
- LoRA paper: https://arxiv.org/abs/2106.09685
- QLoRA paper: https://arxiv.org/abs/2305.14314
- GRPO / DeepSeekMath: https://arxiv.org/abs/2402.03300

Happy fine-tuning.